In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
path_data = "/content/MBA.csv"

data = pd.read_csv(path_data)
data.head()

,application_id,gender,international,gpa,major,race,gmat,work_exp,work_industry,admission
0,1,Female,False,3.30,Business,Asian,620.0,3.0,Financial Services,Admit
1,2,Male,False,3.28,Humanities,Black,680.0,5.0,Investment Management,NaN
2,3,Female,True,3.30,Business,NaN,710.0,5.0,Technology,Admit
3,4,Male,False,3.47,STEM,Black,690.0,6.0,Technology,NaN
4,5,Male,False,3.35,STEM,Hispanic,590.0,5.0,Consulting,NaN


In [3]:
data['admission'] = data['admission'].replace('Admit', 0)
data['admission'] = data['admission'].fillna(1)
data['admission'] = data['admission'].replace('Waitlist', 2)

data.drop(["application_id", "international"], inplace=True, axis=1)
data.dropna(inplace=True, axis=0)

In [4]:
def plot_distribution(column, data=data):
    admitted = data[data['admission'] == 0].drop("admission", axis=1)
    no_admitted = data[data['admission'] == 1].drop("admission", axis=1)
    waitlist = data[data['admission'] == 2].drop("admission", axis=1)

    fig, axs = plt.subplots(1, 3, figsize=(15, 6))

    # Graphic 1: Admitted Students
    admitted_counts = admitted[column].value_counts()
    axs[0].pie(admitted_counts, autopct='%1.1f%%', startangle=90, labels=admitted_counts.index)
    axs[0].set_title(f'{column.capitalize()} Distribution for Admitted Students')

    # Graphic 2: Non-Admitted Students
    no_admitted_counts = no_admitted[column].value_counts()
    axs[1].pie(no_admitted_counts, autopct='%1.1f%%', startangle=90, labels=no_admitted_counts.index)
    axs[1].set_title(f'{column.capitalize()} Distribution for Non-Admitted Students')

    # Graphic 3: Waitlisted Students
    waitlist_counts = waitlist[column].value_counts()
    axs[2].pie(waitlist_counts, autopct='%1.1f%%', startangle=90, labels=waitlist_counts.index)
    axs[2].set_title(f'{column.capitalize()} Distribution for Waitlisted Students')

    plt.tight_layout()
    plt.show()

In [ ]:
column = "race"
plot_distribution(column)

In [ ]:
column = "major"
plot_distribution(column)

In [ ]:
column = "work_exp"
plot_distribution(column)

In [8]:
def plot_data(data):
  for col in data.columns:
    plt.figure(figsize=(12, 8))
    data[col].value_counts().plot(kind='bar')
    plt.title(f"Distribution of {col} for Admitted Students")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
admitted = data[data['admission'] == 0].drop("admission", axis=1)
plot_data(admitted)

In [10]:
# no_admitted = data[data['admission'] == 1].drop("admission", axis=1)
# plot_data(no_admitted)

In [11]:
# waitlist = data[data['admission'] == 2].drop("admission", axis=1)
# plot_data(waitlist)

In [12]:
le = LabelEncoder()
for col in data.columns:
    if data[col].dtype == 'object':
      data[col] = le.fit_transform(data[col])

data = data[data["admission"] != 2]

In [ ]:
corr = data.corr()
correlation_with_admission = corr['admission'].drop('admission').sort_values(ascending=False)
correlation_with_admission.plot(kind='bar');

In [14]:
X = data.drop('admission', axis=1).to_numpy()
y = data['admission'].to_numpy().reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
epochs = 15
batch_size = 32

history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                        validation_data=(X_test, y_test))

_, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest accuracy: {test_accuracy}')

Epoch 1/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6472 - loss: 18.4645 - val_accuracy: 0.8343 - val_loss: 0.4788
Epoch 2/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8588 - loss: 0.4598 - val_accuracy: 0.8343 - val_loss: 0.4645
Epoch 3/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8402 - loss: 0.5149 - val_accuracy: 0.8343 - val_loss: 0.5796
Epoch 4/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8096 - loss: 0.5867 - val_accuracy: 0.8343 - val_loss: 0.5716
Epoch 5/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8384 - loss: 0.5117 - val_accuracy: 0.8343 - val_loss: 0.4851
Epoch 6/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8178 - loss: 0.5910 - val_accuracy: 0.8343 - val_loss: 0.6370
Epoch 7/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8330 - loss: 0.4995 - val_accuracy: 0.8343 - val_loss: 0.4815
Epoch 8/15
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8542 - loss: 0.4499 - val_accuracy: 0

In [17]:
def plot_history(history):
    plt.figure(figsize=(12, 6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

In [ ]:
plot_history(history)